In [1]:
# 创建Agent

## 定义工具函数
## 使用 `@tool` 装饰器，把普通Python函数变成Agent可以调用的工具


from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import SystemMessage

load_dotenv()


True

In [6]:
# 步骤1：用 `@tool` 装饰器定义一个工具
# 函数的文档字符串就是工具的描述
# 模型会根据描述来判断何时调用这个工具

@tool
def get_weather(city: str) -> str:
    '''
    查询指定城市的天气情况
    Args:
        city: 城市名称，如 杭州、北京
    '''
    # 模拟数据演示
    weather_data = {
        "杭州": "晴，25℃，湿度60%",
        "北京": "多云，18℃，湿度45%",
        "上海": "小雨，22℃，湿度80%",
    }

    return weather_data.get(city, f"未找到{city}的天气数据")

@tool
def calculate(expression: str) -> str:
    '''
    执行数学计算。支持加减乘除等基本运算
    Args:
        expression: 数学表达式，如 "3 * 7 + 2
    '''
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果：{expression} = {result}"
    except Exception as e:
        return f"计算错误：{e}"

**文档字符串（函数的""" ... """部分）非常重要。模型会读取工具的描述来决定是否调用这个工具以及传什么参数。描述越清晰，模型就越不容易出错。**

In [7]:
# 步骤2：创建Agent
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 初始化模型
model = init_chat_model("deepseek-v4-flash")

# 创建Agent，传入模型和工具列表
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],     # 工具列表，Agent 可以调用这些工具
    system_prompt="你是一个乐于助人的助手，会使用工具来回答问题。"     # 系统提示词，定义 Agent 的角色和行为
)

In [8]:
# 步骤3：运行Agent
# 构建输入消息
# 消息列表中的第一条通常式 HumanMessage（用户消息）
from langchain.messages import HumanMessage

inputs = {"messages": [HumanMessage(content="杭州天气怎么样？")]}

# invoke() 运行Agent，返回最终状态
result = agent.invoke(inputs)

# 查看消息历史（包含AI的工具调用和工具返回结果）
print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content[:100]}")  # 截取前100字符

print("\n=== 最终回复 ===")
# 最后一条AI消息就是最终答案
print(result["messages"][-1].content)

=== 完整消息历史 ===
[human] 杭州天气怎么样？
[ai] 好的，我马上帮你查询杭州的天气情况！
[tool] 晴，25℃，湿度60%
[ai] 杭州目前的天气情况如下：

🌤 **天气：** 晴  
🌡 **温度：** 25℃  
💧 **湿度：** 60%

天气晴朗，温度舒适，很适合外出活动哦！不过湿度略高，体感可能会稍有点闷。请问还有什

=== 最终回复 ===
杭州目前的天气情况如下：

🌤 **天气：** 晴  
🌡 **温度：** 25℃  
💧 **湿度：** 60%

天气晴朗，温度舒适，很适合外出活动哦！不过湿度略高，体感可能会稍有点闷。请问还有什么可以帮你的吗？😊


In [9]:
# Agent调用多个工具
inputs = {"messages": [HumanMessage(
    content="杭州和北京今天温差多少度？"
)]}
result = agent.invoke(inputs)

print("=== 完整消息历史 ===")
for msg in result["messages"]:
    if msg.type == "tool":
        print(f"[tool {msg.name}] {msg.content}")
    else:
        print(f"[{msg.type}] {msg.content[:120]}")

print("\n=== 最终回复 ===")
print(result["messages"][-1].content)

=== 完整消息历史 ===
[human] 杭州和北京今天温差多少度？
[ai] 好的，我来查询杭州和北京今天的天气情况，然后计算温差。
[tool get_weather] 晴，25℃，湿度60%
[tool get_weather] 多云，18℃，湿度45%
[ai] 好的，我来计算一下温差：
[tool calculate] 计算结果：25 - 18 = 7
[ai] 好的，查询结果如下：

### 🌡️ 杭州 vs 北京 今日温差

| 城市 | 天气 | 温度 |
|:---:|:---:|:---:|
| 🏙️ **杭州** | ☀️ 晴 | **25℃** |
| 🏙️ **北京** | ⛅ 多云

=== 最终回复 ===
好的，查询结果如下：

### 🌡️ 杭州 vs 北京 今日温差

| 城市 | 天气 | 温度 |
|:---:|:---:|:---:|
| 🏙️ **杭州** | ☀️ 晴 | **25℃** |
| 🏙️ **北京** | ⛅ 多云 | **18℃** |

> **温差：7℃**（杭州比北京暖7度）

今天杭州天气晴朗、气温舒适，北京则多云、稍凉一些。如果您要往返两地，记得适当调整着装哦！😊


In [11]:
# 完整代码
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


# 定义工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气情况。

    Args:
        city: 城市名称，如 "杭州"、"北京"
    """
    weather_data = {
        "杭州": "晴，25°C，湿度 60%",
        "北京": "多云，18°C，湿度 45%",
        "上海": "小雨，22°C，湿度 80%",
    }
    return weather_data.get(city, f"未找到 {city} 的天气数据")


@tool
def calculate(expression: str) -> str:
    """执行数学计算。支持加减乘除等基本运算。

    Args:
        expression: 数学表达式，如 "3 * 7 + 2"
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"


# 创建 Agent
model = init_chat_model("deepseek-v4-flash")
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt="你是一个乐于助人的助手，会使用工具来回答问题。",
)

# 运行 Agent
def ask(question: str):
    """发送问题到 Agent 并打印结果"""
    inputs = {"messages": [HumanMessage(content=question)]}
    result = agent.invoke(inputs)
    print(f"问题: {question}")
    print(f"回答: {result['messages'][-1].content}")
    print("-" * 50)
    return result


# 测试几个问题
ask("杭州今天天气怎么样？")
ask("杭州和北京今天温差多少度？")
# 第三个问题：Agent 能够理解"推荐人数"是一个计算问题，并自动调用 calculate 工具。这说明工具调用的决策是由模型对语义的理解来驱动的，而不是硬编码的规则。
ask("菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？")

问题: 杭州今天天气怎么样？
回答: 杭州今天天气不错！☀️

- **天气状况**：晴
- **温度**：25°C
- **湿度**：60%

是个阳光明媚的好天气，温度也很舒适，适合外出活动哦！不过出门记得做好防晒~😊
--------------------------------------------------
问题: 杭州和北京今天温差多少度？
回答: 今天杭州和北京的温差是 **7°C**。

- **杭州**：🌤 晴，25°C
- **北京**：⛅ 多云，18°C
- **温差**：25°C - 18°C = **7°C**

杭州比北京暖和不少，如果从北京去杭州记得少穿点哦～
--------------------------------------------------
问题: 菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？
回答: 没错！计算结果也是 **5**。所以一共有 **5 个人**推荐菜鸟教程 RUNOOB 这个学习平台！🎉

菜鸟教程确实是个很棒的宝藏网站，适合新手入门编程，有 HTML、CSS、Python、Java 等各种教程，感谢你和你朋友们的推荐～ 😄
--------------------------------------------------


{'messages': [HumanMessage(content='菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？', additional_kwargs={}, response_metadata={}, id='befaf955-e188-4d03-8a6e-526823fdf168'),
  AIMessage(content='这是一个简单的数学问题！3 个朋友推荐 + 另外 2 个人推荐 = **5 个人推荐** 👍\n\n让我用计算工具验证一下：', additional_kwargs={'refusal': None, 'reasoning_content': '用户问了一个问题：菜鸟教程 RUNOOB 是一个很棒的学习平台，如果我有3个朋友都推荐了，再加上2个，一共多少人推荐？\n\n这其实是一个简单的数学问题：3 + 2 = 5。\n\n我可以直接回答，不需要调用工具。但用户的问题包含了一个计算，我可以使用计算工具来验证一下。\n\n让我计算一下 3 + 2。'}, response_metadata={'token_usage': {'completion_tokens': 160, 'prompt_tokens': 406, 'total_tokens': 566, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 81, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 150}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_2026

In [13]:
# Agent 支持异步模式，适合在 Web 服务等异步环境中使用
import asyncio
from langchain.messages import HumanMessage


async def main():
    # ainvoke() 是 invoke() 的异步版本
    inputs = {"messages": [HumanMessage(content="杭州天气怎么样？")]}
    result = await agent.ainvoke(inputs)
    print(result["messages"][-1].content)


# 运行异步函数
# asyncio.run(main())
# 直接 await（在Jupyter 中）
result = await main()


杭州今天的天气是**晴天**，气温约为 **25°C**，湿度 **60%**，天气不错，适合出行！☀️


In [ ]:
# LangChain 模型调用 -- init_chat_model() 函数
## 用统一的方式连接 20 多种模型提供商，不需要记忆每个提供商的类名和参数差异。

## init_chat_model() 函数有两种使用模式：

In [ ]:
### 1：固定模型
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

# 指定model模型，返回固定模型
model = init_chat_model('deepseek:deepseek-v4-flash', temperature=0.7)
resp = model.invoke('介绍菜鸟教程 RUNOOB')
print(resp.content)

In [ ]:
### 2：可配置模型
### 不指定 model（或设为 None），创建可在运行时动态切换的模型：
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 不指定 model，返回可配置模型
# 可以固定一些参数（如 temperature=0.7），其余运行时指定
configurable_model = init_chat_model(temperature=0.7)

# 运行时通过 config 指定模型
response = configurable_model.invoke(
    "介绍菜鸟教程 RUNOOB",
    config={"configurable": {"model": "deepseek-v4-flash"}}
)
print(response.content)

# 同一个模型实例，可以用不同的模型来执行
response = configurable_model.invoke(
    "介绍菜鸟教程 RUNOOB",
    config={"configurable": {"model": "claude-sonnet-4-5"}}
)
print(response.content)

常用kwargs参数
kwargs 参数会直接传递给底层模型类，常用的包括：

```{table}
| 参数 |	类型 | 说明 | 适用提供商 |
| temperature |	float |	控制随机性，0~2，默认值因模型而异 | 大部分 |
| max_tokens | int | 限制输出最大 Token 数 | 全部 |
| timeout | int 或 float | 请求超时秒数 | 全部 |
| max_retries | int | 请求失败后的重试次数 | 大部分 |
| base_url | str | 自定义 API 端点 | 大部分 |
| rate_limiter | BaseRateLimiter | 速率限制器实例 | 大部分 |
| top_p | float | 核采样参数，0~1 | 大部分 |
| stop | list[str] | 停止序列，模型遇到这些词时停止生成 | 大部分|
```

ConfigurableModel——运行时切换模型


In [ ]:
from langchain.chat_models import init_chat_model

# 创建可配置模型，并设置默认值
model = init_chat_model(
    "deepseek:deepseek-v4-flash",       # 默认模型
    configurable_fields="any",  # 所有参数都可在运行时修改
    config_prefix="my",         # 配置键前缀
    temperature=0.3,            # 默认温度
)

# 使用默认配置运行
response = model.invoke("介绍菜鸟教程")
print(f"默认配置: {response.content[:50]}...")

# 运行时覆盖模型和参数（注意 my_ 前缀）
response = model.invoke(
    "介绍菜鸟教程 RUNOOB",
    config={
        "configurable": {
            "my_model": "deepseek:deepseek-v4-pro",       # 切换模型
            "my_temperature": 0.9,             # 调整温度
        }
    }
)
print(f"覆盖配置: {response.content[:50]}...")

In [ ]:
# temperature——控制创造性与确定性
# temperature 是最常用的参数，取值范围 0 到 2。它控制模型输出的随机程度。
from langchain.chat_models import init_chat_model

# 同一问题，不同 temperature 的对比
question = "用一句话介绍菜鸟教程 RUNOOB"

# temperature=0：输出非常确定，几乎每次结果一样
model_low = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
resp1 = model_low.invoke(question)
resp2 = model_low.invoke(question)
print(f"temperature=0 第1次: {resp1.content}")
print(f"temperature=0 第2次: {resp2.content}")
print(f"两次结果相同: {resp1.content == resp2.content}")
print()

# temperature=1.5：输出多样化，每次可能不同
model_high = init_chat_model("deepseek:deepseek-v4-flash", temperature=1.5)
resp1 = model_high.invoke(question)
resp2 = model_high.invoke(question)
print(f"temperature=1.5 第1次: {resp1.content}")
print(f"temperature=1.5 第2次: {resp2.content}")


# temperature              值	                        效果	适用场景
# 0 ~ 0.3	    输出稳定、确定，每次结果几乎一致	        数据提取、分类、代码生成、翻译
# 0.5 ~ 0.7	    适度的创造性，输出自然但不偏离主题	    日常对话、内容总结
# 0.8 ~ 1.2	    输出多样化，有较多发挥空间	            创意写作、头脑风暴
# 1.3 ~ 2.0	    输出非常随机，可能出现意外内容	        探索性生成（不太推荐用于生产）



# max_tokens——控制输出长度与成本
# max_tokens 限制模型输出的最大 Token 数。一个 Token 大约相当于 0.75 个英文单词或 0.5 个中文字。

# max_tokens=30：限制输出在 30 个 Token 以内
response_short = model.invoke(
    "详细介绍一下菜鸟教程 RUNOOB 平台",
    max_tokens=30
)
print(f"限制 30 tokens ({len(response_short.content)} 字符):")
print(response_short.content)
print()

# max_tokens=200：允许更长的输出
response_long = model.invoke(
    "详细介绍一下菜鸟教程 RUNOOB 平台",
    max_tokens=200
)
print(f"限制 200 tokens ({len(response_long.content)} 字符):")
print(response_long.content)


# timeout 与 max_retries——网络可靠性
# 在生产环境中，网络请求可能因为各种原因失败。这两个参数帮助你控制请求行为。
# 生产环境推荐配置
model = init_chat_model(
    "deepseek:deepseek-v4-flash",

    # 单次请求最多等待 30 秒
    timeout=30,

    # 失败后最多重试 3 次（总共 4 次请求机会）
    max_retries=3,
)

# 模拟正常调用
try:
    response = model.invoke("菜鸟教程 RUNOOB 是什么？")
    print(f"调用成功: {response.content[:50]}...")
except Exception as e:
    print(f"调用失败: {e}")


# base_url——自定义 API 地址
# base_url 参数在你需要通过代理、中转服务或私有部署访问模型时非常有用。


# 场景 1：通过代理访问 OpenAI
model = init_chat_model(
    "deepseek:deepseek-v4-flash",
    base_url="https://your-proxy-domain.com/v1",  # 代理地址
)

# 场景 2：使用兼容 OpenAI 接口的第三方服务
# 很多国产模型提供了 OpenAI 兼容接口
model = init_chat_model(
    "deepseek:deepseek-v4-flash",           # provider 写 openai
    base_url="https://api.third-party.com/v1",  # 但实际指向第三方
    api_key="your-third-party-key",  # 第三方 API Key
)

# 场景 3：连接本地模型（如 vLLM、Ollama）
model = init_chat_model(
    "openai:qwen2.5",               # 本地模型名
    base_url="http://localhost:8000/v1",  # 本地服务地址
    api_key="not-needed",           # 本地通常不需要 Key
)


# stop——停止序列
model = init_chat_model("deepseek:deepseek-v4-flash")

# stop 参数指定停止序列，模型遇到这些词时会立即停止生成
response = model.invoke(
    "列出五个编程学习网站，每个一行",
    stop=["\n"]  # 遇到换行就停止，只返回第一个
)
print(f"限制 stop=['\\n']: {response.content}")

# LangChain Chat Model 高级用法

In [5]:
# LangChain Chat Model 高级用法
## bind_tools()——让模型知道可以使用哪些工具
## 普通模型只能生成文本。但调用 bind_tools() 后，模型能在回复中返回 tool_call，告诉程序"我需要调用这个工具"。
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "deepseek:deepseek-v4-flash", temperature=0
)

# 用字典描述工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，如 杭州、北京"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# bind_tools() 将工具绑定到模型
# 模型现在“知道”有 get_weather 这个工具可用
model_with_tools = model.bind_tools(tools)

# 问一个需要工具的问题
resp = model_with_tools.invoke("杭州今天天气怎么样？")

# 检查模型是否请求调用工具
if resp.tool_calls:
    print("模型请求调用以下工具：")
    for tc in resp.tool_calls:
        print(f"  工具名: {tc['name']}")
        print(f"  参数: {tc['args']}")
        print(f"  调用ID: {tc['id']}")
else:
    print(f"模型直接回复: {resp.content}")

# 注意模型并没有真正执行 get_weather 函数。bind_tools() 只是告诉模型"你有一个工具可以用"，模型返回的是工具调用的请求。真正的执行由 Agent 或自己编写的代码来完成。

模型请求调用以下工具：
  工具名: get_weather
  参数: {'city': '杭州'}
  调用ID: call_00_y8ji35f1imo71tOab2iU0659


In [7]:
# 用 Pydantic 模型描述工具
# 对于复杂工具，使用 Pydantic 模型定义参数结构比手写字典更清晰：
from pydantic import BaseModel, Field

# 用 Pydantic 定义工具的参数结构
class WeatherInput(BaseModel):
    '''查询指定城市的天气情况'''
    city: str = Field(description="城市名称，如杭州、北京")
    unit: str = Field(
        default="celsius",
        description="温度单位，celsius（摄氏度）或fahrenheit（华氏度）"
    )

class CalculatorInput(BaseModel):
    '''执行数学计算'''
    expression: str = Field(
        description="要计算的数学表达式，如 '(3 + 5) * 2'"
    )

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# 传入 Pydantic 模型，LangChain 自动转换为工具描述
model_with_tools = model.bind_tools([WeatherInput, CalculatorInput])

# 测试复杂场景
resp = model_with_tools.invoke("北京今天多少度？顺便帮我算一下 123 * 456")

print(f"模型请求了 {len(resp.tool_calls)} 个工具调用：")
for tc in resp.tool_calls:
    print(f" {tc['name']}({tc['args']})")

# 使用 Pydantic 定义工具参数是推荐的做法。它提供了类型安全、自动校验，而且 LangChain 会自动从类名和 Field 描述生成工具描述。

模型请求了 2 个工具调用：
 WeatherInput({'city': '北京', 'unit': 'celsius'})
 CalculatorInput({'expression': '123 * 456'})


In [13]:
# with_structured_output()——让模型返回结构化数据
# with_structured_output() 是比 tool_calling 更直接的方式。它让模型按你指定的格式（Schema）返回数据，而不是返回 tool_call。
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# 定义期望的输出结构
class PersonInfo(BaseModel):
    """从文本中提取的人物信息"""
    name: str = Field(description="人物姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")
    skills: list[str] = Field(description="技能列表")

model = init_chat_model("deepseek:deepseek-chat", temperature=0)    # deepseek-v4-flash 存在思考模式，会发生报错

# with_structured_output() 让模型按照 PersonInfo 格式返回
structured_model = model.with_structured_output(PersonInfo)

# 传入非结构化文本，获取结构化数据
text = "张三今年28岁，是一名全栈工程师，精通 Python、React 和 Docker"
result = structured_model.invoke(text)

print(f"姓名: {result.name}")
print(f"年龄: {result.age}")
print(f"职业: {result.occupation}")
print(f"技能: {', '.join(result.skills)}")
print(f"类型: {type(result)}")


# 姓名: 张三
# 年龄: 28
# 职业: 全栈工程师
# 技能: Python, React, Docker
# 类型: <class '__main__.PersonInfo'>

# 返回值直接是 Pydantic 模型实例，可以直接使用 .name、.age 等属性访问。

姓名: 张三
年龄: 28
职业: 全栈工程师
技能: Python, React, Docker
类型: <class '__main__.PersonInfo'>


In [ ]:
# with_structured_output() vs bind_tools()
# 这两个方法看起来相似，但用途不同：
#
#  对比维度  	    with_structured_output()	            bind_tools()
#  用途 	            从文本中提取结构化数据	                让模型知道可用的工具列表
#  返回格式     	    直接返回 Pydantic 对象	            返回 AIMessage，其中包含 tool_calls
#  适用场景 	        信息提取、数据解析	                    Agent 工具调用、需要外部执行的场景
#  模型支持 	        需模型支持原生 structured output	    所有支持 function calling 的模型

In [12]:
# 嵌套结构化输出
# with_structured_output() 支持复杂的嵌套结构：
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class Ingredient(BaseModel):
    """食材信息"""
    name: str = Field(description="食材名称")
    amount: str = Field(description="用量，如 '200g'、'2个'")


class CookingStep(BaseModel):
    """烹饪步骤"""
    step_number: int = Field(description="步骤编号")
    description: str = Field(description="步骤描述")
    duration_minutes: int = Field(description="此步骤需要的时间（分钟）")


class Recipe(BaseModel):
    """菜谱"""
    dish_name: str = Field(description="菜名")
    difficulty: str = Field(description="难度：简单、中等、困难")
    ingredients: list[Ingredient] = Field(description="食材列表")
    steps: list[CookingStep] = Field(description="烹饪步骤")


model = init_chat_model("deepseek:deepseek-chat", temperature=0)    # deepseek-v4-flash 存在思考模式，会发生报错
structured_model = model.with_structured_output(Recipe)

# 输入一个菜谱描述
recipe_text = """
今天来教大家做一道经典的番茄炒蛋，这道菜非常简单。
需要准备：番茄 2 个、鸡蛋 3 个、葱花少许、盐适量、糖少许。
步骤：
1. 先把番茄切块，鸡蛋打散，大概需要 5 分钟
2. 热锅放油，先把鸡蛋炒熟盛出，大概 3 分钟
3. 锅中再放油，炒番茄至出汁，加盐和糖，大概 5 分钟
4. 倒入炒好的鸡蛋，翻炒均匀，撒上葱花，大概 2 分钟
"""

result = structured_model.invoke(recipe_text)

print(f"菜名: {result.dish_name}")
print(f"难度: {result.difficulty}")
print(f"食材 ({len(result.ingredients)} 种):")
for ing in result.ingredients:
    print(f"  - {ing.name}: {ing.amount}")
print(f"步骤 ({len(result.steps)} 步):")
for step in result.steps:
    print(f"  {step.step_number}. {step.description} ({step.duration_minutes}分钟)")

菜名: 番茄炒蛋
难度: 简单
食材 (5 种):
  - 番茄: 2个
  - 鸡蛋: 3个
  - 葱花: 少许
  - 盐: 适量
  - 糖: 少许
步骤 (4 步):
  1. 番茄切块，鸡蛋打散 (5分钟)
  2. 热锅放油，先把鸡蛋炒熟盛出 (3分钟)
  3. 锅中再放油，炒番茄至出汁，加盐和糖 (5分钟)
  4. 倒入炒好的鸡蛋，翻炒均匀，撒上葱花 (2分钟)


In [14]:
from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek:deepseek-chat", temperature=0)

# 直接传入 JSON Schema
json_schema = {
    "title": "SentimentAnalysis",
    "description": "情感分析结果",
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["positive", "negative", "neutral"],
            "description": "情感倾向"
        },
        "confidence": {
            "type": "number",
            "description": "置信度，0~1"
        },
        "keywords": {
            "type": "array",
            "items": {"type": "string"},
            "description": "关键情感词"
        }
    },
    "required": ["sentiment", "confidence"]
}

structured_model = model.with_structured_output(json_schema)
result = structured_model.invoke("菜鸟教程 RUNOOB 真的太棒了，强烈推荐给所有编程新手！")

print(f"情感: {result['sentiment']}")
print(f"置信度: {result['confidence']}")
print(f"关键词: {result['keywords']}")

情感: positive
置信度: 0.95
关键词: ['菜鸟教程', 'RUNOOB', '太棒了', '强烈推荐', '编程新手']


In [ ]:
# ConfigurableModel 上的 bind_tools 和 with_structured_output
# 可配置模型也支持这两个方法，用法完全相同：
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# 创建可配置模型
configurable_model = init_chat_model(
    "deepseek-v4-flash",
    configurable_fields=("model", "model_provider"),
    temperature=0,
)

# 链式调用：先绑定工具，再调用
configurable_with_tools = configurable_model.bind_tools([...])

# 运行时可以用不同模型来执行
result = configurable_with_tools.invoke(
    "查询天气",
    config={"configurable": {"model": "claude-sonnet-4-5"}}
)

# 在 ConfigurableModel 上链式调用 bind_tools 或 with_structured_output 时，实际操作会被延迟执行——直到模型实例化时才真正绑定，因此不会影响运行时动态切换模型的功能。

# LangChain 消息类型

In [ ]:
# 四种核心消息类型
# LangChain 定义了四种核心消息类型，分别对应对话中的不同角色：
#
# 类型	            角色	            说明	                                典型内容
# HumanMessage	    用户	        用户发送的消息	                        "今天天气怎么样？"
# AIMessage	        AI 助手	    模型的回复，可能包含 tool_calls	        "今天杭州晴天，25°C"
# SystemMessage	    系统	        系统指令，定义 AI 的角色和行为规则	    "你是一个专业的天气助手"
# ToolMessage	    工具	        工具执行后的返回结果	                "晴，25°C，湿度 60%"

In [17]:
# HumanMessage——用户消息
# HumanMessage 代表用户发送给 AI 的消息。它是最常见的消息类型，也是对话的起点。

from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

# 创建一条用户消息
msg = HumanMessage(content="菜鸟教程 RUNOOB 是什么？")

print(f"类型：{msg.type}")         # human
print(f"内容：{msg.content}")      # 菜鸟教程 RUNOOB 是什么？

# 创建消息列表（代表多轮对话历史）
messages = [
    HumanMessage(content="你好"),
    HumanMessage(content="菜鸟教程有哪些课程？"),
    HumanMessage(content="Python 课程适合零基础吗？")
]

model = init_chat_model("deepseek:deepseek-chat", temperature=0)
response = model.invoke(messages)
print(f"\n模型回复：{response.content}")

类型：human
内容：菜鸟教程 RUNOOB 是什么？

模型回复：你好！很高兴为你解答。

**菜鸟教程（runoob.com）** 是一个非常受欢迎的编程入门网站，它提供了大量**免费、简洁、实用**的教程。主要课程涵盖：

1.  **前端开发**：HTML、CSS、JavaScript、jQuery、Vue.js、React、Angular、Bootstrap 等。
2.  **后端开发**：Python、Java、PHP、C、C++、C#、Go、Ruby、Node.js、Perl 等。
3.  **数据库**：MySQL、SQL Server、PostgreSQL、MongoDB、Redis、SQLite 等。
4.  **移动开发**：Android、iOS、Swift、Kotlin 等。
5.  **服务器与运维**：Linux、Docker、Git、Nginx、Apache 等。
6.  **其他**：正则表达式、XML、JSON、Markdown 等。

---

**关于 Python 课程是否适合零基础？**

**非常适合。** 菜鸟教程的 Python 课程是专门为**零基础**学习者设计的，原因如下：

-   **从最基础讲起**：它会从“安装 Python”、“第一个程序（Hello World）”、“变量”、“数据类型”开始，完全不需要任何编程背景。
-   **内容简洁明了**：每个知识点都配有简单的代码示例，没有太多复杂的理论堆砌，非常适合快速上手。
-   **在线运行代码**：网站内置了在线代码编辑器，你可以直接修改和运行示例代码，边学边练，效果很好。
-   **覆盖核心知识点**：包括基础语法、流程控制、函数、模块、文件操作、面向对象编程等，学完后能打下扎实的基础。

**建议**：如果你是纯小白，可以先跟着菜鸟教程的 Python 基础部分走一遍，配合动手敲代码，很快就能入门。之后如果想深入学习，可以再结合其他书籍或项目实战。

**总结：菜鸟教程的 Python 课程是零基础入门的一个绝佳起点。**


In [ ]:
# HumanMessage 的快捷创建方式
# 在构建消息列表时，可以使用元组或字典作为快捷方式：

from langchain.messages import HumanMessage

# 方式1：标准构造
msg1 = HumanMessage(content="你好")

# 方式2：元组快捷方式（role，content）
msg2 = ("user", "你好")
msg3 = ("human", "你好")

# 方式3：字典快捷方式
msg4 = {"role": "user", "content": "你好"}

# 四种方式等价，都会在 Agent 内部被转换为 HumanMessage
print(type(msg1))  # <class 'langchain_core.messages.human.HumanMessage'>

In [18]:
# AIMessage——AI 回复
# AIMessage 代表模型的回复。与普通文本不同，AIMessage 可能包含 tool_calls（工具调用请求）。

from langchain.messages import AIMessage

# 普通AI回复（无工具调用）
ai_msg = AIMessage(content="菜鸟教程是一个编程学习平台")

# 包含工具调用的AI回复
ai_with_tools = AIMessage(
    content="",     # 工具调用时 content 通常为空
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"city": "杭州"},
            "id": "call_abc123",
            "type": "tool_call",
        }
    ]
)

print("=== 普通 AI 消息 ===")
print(f"content: {ai_msg.content}")
print(f"tool_calls: {ai_msg.tool_calls}")   # []

print("\n=== 含工具调用的 AI 消息 ===")
print(f"content: {ai_with_tools.content}")
print(f"tool_calls: {ai_with_tools.tool_calls}")
# [{'name': 'get_weather', 'args': {'city': '杭州'}, ...}]

=== 普通 AI 消息 ===
content: 菜鸟教程是一个编程学习平台
tool_calls: []

=== 含工具调用的 AI 消息 ===
content: 
tool_calls: [{'name': 'get_weather', 'args': {'city': '杭州'}, 'id': 'call_abc123', 'type': 'tool_call'}]


In [19]:
# AIMessage 的附加信息

from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek:deepseek-chat")
response = model.invoke("介绍菜鸟教程 RUNOOB")

# AIMessage 包含丰富的元数据
print(f"内容: {response.content}")
print(f"消息ID: {response.id}")
print(f"模型名: {response.response_metadata.get('model_name')}")
print(f"完成原因: {response.response_metadata.get('finish_reason')}")

# usage_metadata 包含 Token 用量信息
if response.usage_metadata:
    print(f"输入 tokens: {response.usage_metadata.get('input_tokens')}")
    print(f"输出 tokens: {response.usage_metadata.get('output_tokens')}")
    print(f"总计 tokens: {response.usage_metadata.get('total_tokens')}")

内容: 菜鸟教程（RUNOOB）是一个非常知名的中文编程学习网站，主要面向编程初学者和需要快速查阅技术文档的开发者。以下是它的详细介绍：

### 1. 核心定位
- **服务对象**：零基础编程入门者、学生、转行人员以及需要快速查阅语法或例子的工程师。
- **核心理念**：提供“**最简单、最易懂**”的入门教程，号称“学的不仅是技术，更是梦想”。

### 2. 主要特点
- **内容覆盖面广**：几乎涵盖了所有主流编程语言和技术栈：
  - **前端**：HTML/CSS/JavaScript、Vue.js、React、jQuery、Bootstrap
  - **后端**：Python、Java、PHP、Go、Ruby、Node.js、C/C++、C#
  - **数据库**：MySQL、SQL Server、MongoDB、Redis、SQLite
  - **数据处理**：Pandas、NumPy、Matplotlib、Scipy
  - **工具与系统**：Git、Linux、Docker、正则表达式、Markdown
  - **服务器/接口**：Apache、Nginx、JSON、XML、AJAX
  - **移动端/其他**：Swift、Kotlin、Android、iOS 基础
- **免费与易用**：所有教程完全免费，无需注册登录即可学习。
- **在线工具集**：提供大量的在线测试工具，如：
  - 在线代码编辑器（编译运行代码）
  - HTML/CSS/JS 在线测试
  - JSON/XML/正则表达式 格式化与验证
  - 时间戳转换、颜色值转换、二维码生成等
- **典型案例**：每个知识点通常配有可直接运行的代码示例，方便复制修改。
- **离线学习**：支持离线版下载，方便无网络环境学习。

### 3. 优缺点分析

| 优势 | 劣势 |
|------|------|
| ✅ 内容清晰、结构简单，适合零基础 | ❌ 内容较为浅显，难以深入高级主题 |
| ✅ 收录了“菜鸟必读”的经典入门路径 | ❌ 部分教程更新不及时（尤其对于快速演进的框架如React） |
| ✅ 方便快捷的在线工具和测试环境 | ❌ 缺少系统性的项目实战和练习题 |
| ✅ 中文化程度高，对英文不好的用户友好 | ❌ 代码示例有时偏旧或不够规范 

In [ ]:
# SystemMessage——系统指令
# SystemMessage 用于设定 AI 的行为、角色和约束。它放在消息列表的最前面，指导模型如何回复。

from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0.7)

# 没有系统指定的回复
messages_no_system = [HumanMessage(content="介绍菜鸟教程")]
response = model.invoke(messages_no_system, max_tokens=30)
print(f"无系统指令：{response.content[:80]}")

# 有系统指定的回复
messages_with_system = [
    SystemMessage(content="你是一个小红书风格的博主，回复要活泼、使用 emoji、带话题标签"),
    HumanMessage(content="介绍菜鸟教程")
]
response = model.invoke(messages_with_system)
print(f"\n有系统指令: {response.content}")